# Getting Steady State

## intro
This code implements the steady state solution for various political regimes. Some things to note:

- Each steady state is "regime-specific," which means that individuals are not considering other regimes or even the possibilty of transitioning to another regime
- This is all in Julia, and I'm falling apart trying to code it
- 

To run this code, simply click "Run." I think, honestly this is for replicability but I always assumed Jupyter was for suckers until I realized this is 10x better than my "self-documenting code."

## directory structure

I grew up oop-ing, so this is a result of that. All my functions are stored in the relevant module. 

```
populism/
├── src/                        # your modules
│   ├── ModelTypes.jl
│   ├── ModelFunctions.jl
│   ├── Compute.jl
│   ├── EGM.jl
│   ├── DistTools.jl
│   └── Solvers.jl
├── PopulismModel.jl            # parent module, includes everything
├── notebooks/
│   └── figures.ipynb           # if you want plots separate
└── Utitled.ipynb              # primary replication notebook, this one, for now.
```

Running `PopulismModel.jl` imports all the necessary functions, and `ModelTypes.jl` defines the structres that I'll be passing in between functions. 

## running

Importing all the necessary methods below. 

In [1]:

# run if you don't have this package (I didn't), otherwise comment out
# import Pkg; Pkg.add("Printf")

using Printf
using LinearAlgebra: dot

include("src/ModelTypes.jl")
@printf("Loaded ModelTypes.jl\n")

include("src/Compute.jl")
@printf("Loaded Compute.jl\n")

include("src/ModelFunctions.jl")
@printf("Loaded ModelFunctions.jl\n")

include("src/EGM.jl")
@printf("Loaded EGM.jl\n")

include("src/DistrTools.jl")
@printf("Loaded DistrTools.jl\n")

include("src/Solvers.jl")
@printf("Loaded Solvers.jl\n")

include("src/SteadyState.jl")
@printf("Loaded Steady States.jl\n")

using .ModelTypes
using .ModelFunctions
using .Compute
using .EGM
using .DistrTools
using .Solvers
using .SteadyState


Loaded ModelTypes.jl
Loaded Compute.jl
Loaded ModelFunctions.jl
Loaded EGM.jl
Loaded DistrTools.jl
Loaded Solvers.jl
Loaded Steady States.jl


Putting in the calibrations that I'll throw through 

In [2]:
# model parameters
const α::Float64 = 0.36;
const β::Float64 = 0.96;
const δ::Float64 = 0.06;
const σ::Float64 = 2;
const ϕ::Float64 = 0;

# grid sizes and parameters
const na::Int64 = 100; 
const nl::Int64 = 7;
const nz::Int64 = 2;

const a_l::Float64 = 0;
const a_h::Float64 = 100;

# calibrations for idiosyncratic income (vibes)
const μ_l::Float64 = 0; 
const ρ_l::Float64 = .9;
const σ_l::Float64 = .2;

# calibrations for aggregate TFP (Khan and Thomas 2013 would have 
# ρ_z = 0.909; I am setting it lower for two dimensional z with
# some variance for now.)
const μ_z::Float64 = 0;
const ρ_z::Float64 = .75;
const σ_z::Float64 = 0.014;

Now getting the transition matrices for idiosyncratic productivity ($\varepsilon$) and aggregate TFP ($z$). We'll also construct the savings grid, which is logspaced. 

In [3]:

# need to back out σ^2_e given σ^2_l
σ_le = σ_l*sqrt(1 - ρ_l^2);
grid_range = 2.575;

π_l, lgrid = getTauchen(nl,  μ_l, σ_le, ρ_l, grid_range);

stationary_l = stationary(π_l);
const lagg::Float64 = dot(stationary_l, lgrid);

σ_ze = σ_z*sqrt(1 - ρ_z^2);
π_z, zgrid= getTauchen(nz,  μ_z, σ_ze, ρ_z, grid_range);

agrid = logspace(a_l, a_h, na);
amu = collect(range(a_l, a_h, length=na*10));

Now that we have all the grids, we can initiate the model params into a single object I can pass into the steady state process: 

In [4]:
const params = ModelParams(α, β, δ, σ, ϕ, agrid, 
    lgrid, zgrid, π_l, π_z, amu);

Now we move on to defining the government problem. Because we want this to be solving a ton of steady states across various combinations of regime policies, what I'm going to do is linearly space out the policies:
- $\tau$: no progressivity (0) to fully redistributive (1);
- $\eta$: no immigration (0) to full addition of population in the next year (1).

Then we'll loop through these regime combinations and solve for the steady state in each combination.

In [5]:
const np::Int64 = 10; # number of policies
const pol_l::Float64 = 0;
const pol_h::Float64 = 1;

τ_grid = range(pol_l, pol_h, length = np);
η_grid = range(pol_l, pol_h, length = np);

captax = repeat([0], outer = 7);

In [6]:
policies = ProposedPolicies(.1, .1, captax)

ProposedPolicies(0.1, 0.1, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

It's now time to loop (I hope). I'm going to, for each policy combination, solve for steady state by: 
1. Making a guess at the aggregate capital demanded on the market (`K_val`)
2. Calculate the implied interest rate (`r`) and wages (`w`) given aggregate labor (`lagg`) multiplied by the policy coefficient for migrants (`\eta`).
3. Check, after solving the HH problem, how much capital would be supplied at this rate.
4. If it's off, adjust guess by a binary search between new bounds and try again. 

In [7]:
kl::Float64 = 0; kh::Float64 = 20; 
kval::Float64 = (kl + kh)/2;

kdist::Float64 = 1e5

vTol::Float64 = 1e-6

while kdist > vTol
    

    V, G, C, CI, LI = solveHousehold(params, policies, kval, vTol);

    μ, K = getDistr(G, amu, agrid, π_l, π_z, CI, LI, ϕ, verbose = true);
    
    diff = K - kval
    adj = abs(diff) < 10 ? 0.5 : 0.7 # fancy if-then in one line

    if diff > 0
        @printf("\n||K - kval|| = %4.5f. \tCapital too low.\n", abs(diff))
        kl = (1 - adj) * kval + adj * kl
    else
        @printf("\n||K - kval|| = %4.5f. \tCapital too high.\n", abs(diff))
        kh = (1 - adj) * kval + adj * kh
    end

    kdist = abs(diff)
    kval = 0.5 * (kl + kh)
    
end

Solving Household Problem...Iteration 50: ||V - V0|| = 0.143506, ||G - G0|| = 0.000000, dist = 0.287012
Iteration 100: ||V - V0|| = 0.050861, ||G - G0|| = 0.000000, dist = 0.101722
Iteration 150: ||V - V0|| = 0.018436, ||G - G0|| = 0.000000, dist = 0.036872
Iteration 200: ||V - V0|| = 0.006697, ||G - G0|| = 0.000000, dist = 0.013393
Iteration 250: ||V - V0|| = 0.002434, ||G - G0|| = 0.000000, dist = 0.004867
Iteration 300: ||V - V0|| = 0.000884, ||G - G0|| = 0.000000, dist = 0.001769
Iteration 350: ||V - V0|| = 0.000322, ||G - G0|| = 0.000000, dist = 0.000643
Iteration 400: ||V - V0|| = 0.000117, ||G - G0|| = 0.000000, dist = 0.000234
Iteration 450: ||V - V0|| = 0.000043, ||G - G0|| = 0.000000, dist = 0.000085
Iteration 500: ||V - V0|| = 0.000015, ||G - G0|| = 0.000000, dist = 0.000031
Iteration 550: ||V - V0|| = 0.000006, ||G - G0|| = 0.000000, dist = 0.000011
Iteration 600: ||V - V0|| = 0.000002, ||G - G0|| = 0.000000, dist = 0.000004
Iteration 650: ||V - V0|| = 0.000001, ||G - G0|| 


		Iteration 696: ||Tm - m|| = 0.000001	sum = 1.0000

||K - kval|| = 5.46454. 	Capital too low.
Solving Household Problem...Iteration 50: ||V - V0|| = 0.148028, ||G - G0|| = 0.000000, dist = 0.296055
Iteration 100: ||V - V0|| = 0.052347, ||G - G0|| = 0.000000, dist = 0.104695
Iteration 150: ||V - V0|| = 0.018949, ||G - G0|| = 0.000000, dist = 0.037899
Iteration 200: ||V - V0|| = 0.006878, ||G - G0|| = 0.000000, dist = 0.013755
Iteration 250: ||V - V0|| = 0.002498, ||G - G0|| = 0.000000, dist = 0.004996
Iteration 300: ||V - V0|| = 0.000908, ||G - G0|| = 0.000000, dist = 0.001815
Iteration 350: ||V - V0|| = 0.000330, ||G - G0|| = 0.000000, dist = 0.000660
Iteration 400: ||V - V0|| = 0.000120, ||G - G0|| = 0.000000, dist = 0.000240
Iteration 450: ||V - V0|| = 0.000044, ||G - G0|| = 0.000000, dist = 0.000087
Iteration 500: ||V - V0|| = 0.000016, ||G - G0|| = 0.000000, dist = 0.000032
Iteration 550: ||V - V0|| = 0.000006, ||G - G0|| = 0.000000, dist = 0.000012
Iteration 600: ||V - V0|| = 0.

Iteration 650: ||V - V0|| = 0.000001, ||G - G0|| = 0.000000, dist = 0.000002
Iteration 673: ||V - V0|| = 0.000000, ||G - G0|| = 0.000000, dist = 0.000001

		Iteration 734: ||Tm - m|| = 0.000001	sum = 1.0000

||K - kval|| = 0.29755. 	Capital too high.
Solving Household Problem...Iteration 50: ||V - V0|| = 0.150944, ||G - G0|| = 0.000000, dist = 0.301888
Iteration 100: ||V - V0|| = 0.053270, ||G - G0|| = 0.000000, dist = 0.106540
Iteration 150: ||V - V0|| = 0.019251, ||G - G0|| = 0.000000, dist = 0.038502
Iteration 200: ||V - V0|| = 0.006978, ||G - G0|| = 0.000000, dist = 0.013956
Iteration 250: ||V - V0|| = 0.002532, ||G - G0|| = 0.000000, dist = 0.005063
Iteration 300: ||V - V0|| = 0.000919, ||G - G0|| = 0.000000, dist = 0.001838
Iteration 350: ||V - V0|| = 0.000334, ||G - G0|| = 0.000000, dist = 0.000667
Iteration 400: ||V - V0|| = 0.000121, ||G - G0|| = 0.000000, dist = 0.000242
Iteration 450: ||V - V0|| = 0.000044, ||G - G0|| = 0.000000, dist = 0.000088
Iteration 500: ||V - V0|| = 0

Iteration 550: ||V - V0|| = 0.000006, ||G - G0|| = 0.000000, dist = 0.000012
Iteration 600: ||V - V0|| = 0.000002, ||G - G0|| = 0.000000, dist = 0.000004
Iteration 650: ||V - V0|| = 0.000001, ||G - G0|| = 0.000000, dist = 0.000002
Iteration 673: ||V - V0|| = 0.000000, ||G - G0|| = 0.000000, dist = 0.000001

		Iteration 726: ||Tm - m|| = 0.000001	sum = 1.0000

||K - kval|| = 0.35180. 	Capital too low.
Solving Household Problem...Iteration 50: ||V - V0|| = 0.150486, ||G - G0|| = 0.000000, dist = 0.300971
Iteration 100: ||V - V0|| = 0.053128, ||G - G0|| = 0.000000, dist = 0.106255
Iteration 150: ||V - V0|| = 0.019206, ||G - G0|| = 0.000000, dist = 0.038411
Iteration 200: ||V - V0|| = 0.006964, ||G - G0|| = 0.000000, dist = 0.013927
Iteration 250: ||V - V0|| = 0.002527, ||G - G0|| = 0.000000, dist = 0.005054
Iteration 300: ||V - V0|| = 0.000917, ||G - G0|| = 0.000000, dist = 0.001835
Iteration 350: ||V - V0|| = 0.000333, ||G - G0|| = 0.000000, dist = 0.000666
Iteration 400: ||V - V0|| = 0.

Iteration 450: ||V - V0|| = 0.000044, ||G - G0|| = 0.000000, dist = 0.000088
Iteration 500: ||V - V0|| = 0.000016, ||G - G0|| = 0.000000, dist = 0.000032
Iteration 550: ||V - V0|| = 0.000006, ||G - G0|| = 0.000000, dist = 0.000012
Iteration 600: ||V - V0|| = 0.000002, ||G - G0|| = 0.000000, dist = 0.000004
Iteration 650: ||V - V0|| = 0.000001, ||G - G0|| = 0.000000, dist = 0.000002
Iteration 673: ||V - V0|| = 0.000000, ||G - G0|| = 0.000000, dist = 0.000001

		Iteration 725: ||Tm - m|| = 0.000001	sum = 1.0000

||K - kval|| = 0.00668. 	Capital too high.
Solving Household Problem...Iteration 50: ||V - V0|| = 0.150496, ||G - G0|| = 0.000000, dist = 0.300992
Iteration 100: ||V - V0|| = 0.053131, ||G - G0|| = 0.000000, dist = 0.106262
Iteration 150: ||V - V0|| = 0.019207, ||G - G0|| = 0.000000, dist = 0.038413
Iteration 200: ||V - V0|| = 0.006964, ||G - G0|| = 0.000000, dist = 0.013928
Iteration 250: ||V - V0|| = 0.002527, ||G - G0|| = 0.000000, dist = 0.005054
Iteration 300: ||V - V0|| = 0

Iteration 350: ||V - V0|| = 0.000333, ||G - G0|| = 0.000000, dist = 0.000666
Iteration 400: ||V - V0|| = 0.000121, ||G - G0|| = 0.000000, dist = 0.000242
Iteration 450: ||V - V0|| = 0.000044, ||G - G0|| = 0.000000, dist = 0.000088
Iteration 500: ||V - V0|| = 0.000016, ||G - G0|| = 0.000000, dist = 0.000032
Iteration 550: ||V - V0|| = 0.000006, ||G - G0|| = 0.000000, dist = 0.000012
Iteration 600: ||V - V0|| = 0.000002, ||G - G0|| = 0.000000, dist = 0.000004
Iteration 650: ||V - V0|| = 0.000001, ||G - G0|| = 0.000000, dist = 0.000002
Iteration 673: ||V - V0|| = 0.000000, ||G - G0|| = 0.000000, dist = 0.000001

		Iteration 725: ||Tm - m|| = 0.000001	sum = 1.0000

||K - kval|| = 0.00616. 	Capital too high.
Solving Household Problem...Iteration 50: ||V - V0|| = 0.150488, ||G - G0|| = 0.000000, dist = 0.300977
Iteration 100: ||V - V0|| = 0.053128, ||G - G0|| = 0.000000, dist = 0.106257
Iteration 150: ||V - V0|| = 0.019206, ||G - G0|| = 0.000000, dist = 0.038412
Iteration 200: ||V - V0|| = 0

Iteration 250: ||V - V0|| = 0.002527, ||G - G0|| = 0.000000, dist = 0.005054
Iteration 300: ||V - V0|| = 0.000917, ||G - G0|| = 0.000000, dist = 0.001835
Iteration 350: ||V - V0|| = 0.000333, ||G - G0|| = 0.000000, dist = 0.000666
Iteration 400: ||V - V0|| = 0.000121, ||G - G0|| = 0.000000, dist = 0.000242
Iteration 450: ||V - V0|| = 0.000044, ||G - G0|| = 0.000000, dist = 0.000088
Iteration 500: ||V - V0|| = 0.000016, ||G - G0|| = 0.000000, dist = 0.000032
Iteration 550: ||V - V0|| = 0.000006, ||G - G0|| = 0.000000, dist = 0.000012
Iteration 600: ||V - V0|| = 0.000002, ||G - G0|| = 0.000000, dist = 0.000004
Iteration 650: ||V - V0|| = 0.000001, ||G - G0|| = 0.000000, dist = 0.000002
Iteration 673: ||V - V0|| = 0.000000, ||G - G0|| = 0.000000, dist = 0.000001

		Iteration 725: ||Tm - m|| = 0.000001	sum = 1.0000

||K - kval|| = 0.00025. 	Capital too low.
Solving Household Problem...Iteration 50: ||V - V0|| = 0.150488, ||G - G0|| = 0.000000, dist = 0.300977
Iteration 100: ||V - V0|| = 0.

Iteration 150: ||V - V0|| = 0.019206, ||G - G0|| = 0.000000, dist = 0.038412
Iteration 200: ||V - V0|| = 0.006964, ||G - G0|| = 0.000000, dist = 0.013927
Iteration 250: ||V - V0|| = 0.002527, ||G - G0|| = 0.000000, dist = 0.005054
Iteration 300: ||V - V0|| = 0.000917, ||G - G0|| = 0.000000, dist = 0.001835
Iteration 350: ||V - V0|| = 0.000333, ||G - G0|| = 0.000000, dist = 0.000666
Iteration 400: ||V - V0|| = 0.000121, ||G - G0|| = 0.000000, dist = 0.000242
Iteration 450: ||V - V0|| = 0.000044, ||G - G0|| = 0.000000, dist = 0.000088
Iteration 500: ||V - V0|| = 0.000016, ||G - G0|| = 0.000000, dist = 0.000032
Iteration 550: ||V - V0|| = 0.000006, ||G - G0|| = 0.000000, dist = 0.000012
Iteration 600: ||V - V0|| = 0.000002, ||G - G0|| = 0.000000, dist = 0.000004
Iteration 650: ||V - V0|| = 0.000001, ||G - G0|| = 0.000000, dist = 0.000002
Iteration 673: ||V - V0|| = 0.000000, ||G - G0|| = 0.000000, dist = 0.000001

		Iteration 725: ||Tm - m|| = 0.000001	sum = 1.0000

||K - kval|| = 0.00013

Iteration 100: ||V - V0|| = 0.053128, ||G - G0|| = 0.000000, dist = 0.106257
Iteration 150: ||V - V0|| = 0.019206, ||G - G0|| = 0.000000, dist = 0.038412
Iteration 200: ||V - V0|| = 0.006964, ||G - G0|| = 0.000000, dist = 0.013927
Iteration 250: ||V - V0|| = 0.002527, ||G - G0|| = 0.000000, dist = 0.005054
Iteration 300: ||V - V0|| = 0.000917, ||G - G0|| = 0.000000, dist = 0.001835
Iteration 350: ||V - V0|| = 0.000333, ||G - G0|| = 0.000000, dist = 0.000666
Iteration 400: ||V - V0|| = 0.000121, ||G - G0|| = 0.000000, dist = 0.000242
Iteration 450: ||V - V0|| = 0.000044, ||G - G0|| = 0.000000, dist = 0.000088
Iteration 500: ||V - V0|| = 0.000016, ||G - G0|| = 0.000000, dist = 0.000032
Iteration 550: ||V - V0|| = 0.000006, ||G - G0|| = 0.000000, dist = 0.000012
Iteration 600: ||V - V0|| = 0.000002, ||G - G0|| = 0.000000, dist = 0.000004
Iteration 650: ||V - V0|| = 0.000001, ||G - G0|| = 0.000000, dist = 0.000002
Iteration 673: ||V - V0|| = 0.000000, ||G - G0|| = 0.000000, dist = 0.000001